In [2]:
import os
import django

# Replace 'myproject' with your actual Django project directory name
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "datamodel_demo.settings")

# Prevents synchronous operation errors in the asynchronous Jupyter loop
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

django.setup()

In [3]:
from app1.models import Entity, EntityType
from app1.utils import (
    descendant_types, entries_of_type, group_by_type, count_by_type,
    traverse_entities, latest_information_record,
)

In [6]:
EntityType.objects.all()[0].code

'material_entity'

In [7]:
from django.db.models import Count
from IPython.display import display
from app1.models import (
    Entity, EntityInformationRecord, EntityType, EntityTypeRecordSlot,
    InformationRecordType,
)

print("Entity types:", EntityType.objects.count())
print("Information-record types:", InformationRecordType.objects.count())
print("Entities:", Entity.objects.count())
print("Entity information records:", EntityInformationRecord.objects.count())

print("\nEntityType overview")
display(list(
    EntityType.objects.annotate(entity_count=Count("entities"))
    .order_by("code")
    .values("code", "name", "is_instantiable", "entity_count")
))

print("\nInformationRecordType overview")
display(list(
    InformationRecordType.objects.annotate(record_count=Count("entityinformationrecord"))
    .order_by("code")
    .values("code", "name", "is_instantiable", "record_count")
))

Entity types: 1
Information-record types: 0
Entities: 7
Entity information records: 7

EntityType overview


[{'code': 'material_entity',
  'name': 'Material entity',
  'is_instantiable': True,
  'entity_count': 7}]


InformationRecordType overview


[]

## Type hierarchies

Inspect the roots and descendants in each tier-2 vocabulary. The shared utility handles descendant traversal safely.

In [8]:
def hierarchy_rows(model):
    rows = []
    for root in model.objects.filter(parent__isnull=True).order_by("code"):
        for node in descendant_types(root):
            depth = 0
            parent = node.parent
            while parent is not None:
                depth += 1
                parent = parent.parent
            rows.append({
                "code": node.code,
                "name": node.name,
                "depth": depth,
                "parent": node.parent.code if node.parent else None,
                "is_instantiable": node.is_instantiable,
            })
    return rows

print("EntityType hierarchy")
display(hierarchy_rows(EntityType))

print("InformationRecordType hierarchy")
display(hierarchy_rows(InformationRecordType))

EntityType hierarchy


[{'code': 'material_entity',
  'name': 'Material entity',
  'depth': 0,
  'parent': None,
  'is_instantiable': True}]

InformationRecordType hierarchy


[]

## Record coverage and constraints

Compare current information-record usage with the type vocabulary and inspect the entity-type rules that describe expected sidecars.

In [9]:
record_usage = list(
    EntityInformationRecord.objects
    .values("information_record_type__code")
    .annotate(record_count=Count("pk"), entity_count=Count("entity", distinct=True))
    .order_by("information_record_type__code")
)

print("Entity information-record usage")
display(record_usage)

print("Records without a type:", EntityInformationRecord.objects.filter(
    information_record_type__isnull=True
).count())

print("EntityTypeRecordSlot rules")
display(list(
    EntityTypeRecordSlot.objects.select_related("entity_type", "record_type")
    .order_by("entity_type__code", "record_type__code")
    .values(
        "entity_type__code", "record_type__code",
        "min_count", "max_count", "match_mode",
    )
))

Entity information-record usage


[{'information_record_type__code': None, 'record_count': 7, 'entity_count': 7}]

Records without a type: 7
EntityTypeRecordSlot rules


[]